# Server-Telemetry Anomaly Detection

This notebook audits telemetry, confirms that labels are evaluation-only, executes the detector comparison, inspects thresholded errors, and reconciles the selected method with exported artifacts.

## 1. Setup and telemetry audit

The anomaly label is retained only for evaluation. It is not used to fit the unsupervised detectors.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'server_telemetry.csv'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts'
df = pd.read_csv(DATA_PATH)
signal_columns = [column for column in df.columns if column != 'anomaly']
audit = {'shape': df.shape, 'signals': len(signal_columns), 'missing_values': int(df.isna().sum().sum()), 'duplicate_rows': int(df.duplicated().sum()), 'anomaly_rate': round(float(df['anomaly'].mean()), 4)}
display(df.head())
audit

## 2. Execute the reproducible experiment

The command compares Isolation Forest, Local Outlier Factor, and One-Class SVM using the documented contamination and evaluation setup.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, str(PROJECT_ROOT / 'src' / 'run_experiment.py'), '--data', str(DATA_PATH), '--output', str(OUTPUT_DIR)], check=True)

## 3. Detector comparison and visual evidence

PR-AUC is emphasized because anomalies are rare. False-positive rate, precision, recall, and F1 are reviewed with ranking metrics.

In [ ]:
comparison = pd.read_csv(OUTPUT_DIR / 'model-comparison.csv')
display(comparison)
display(Image(filename=str(OUTPUT_DIR / 'model-comparison.png')))
display(Image(filename=str(OUTPUT_DIR / 'selected-confusion-matrix.png')))
display(pd.read_csv(OUTPUT_DIR / 'anomaly-feature-deviations.csv'))

## 4. Reconcile outputs

The JSON summary is authoritative for label usage, selected method, metrics, artifact names, and limitations.

In [ ]:
summary = json.loads((OUTPUT_DIR / 'results-summary.json').read_text())
assert summary['labels_used_for_training'] is False
assert summary['selected_method'] == comparison.iloc[0]['method']
summary

## Interpretation

An alert is a signal for investigation, not proof of an attack or system failure. Deployment would require analyst feedback, alert-volume limits, threshold review, and drift monitoring.